# Temporal merge-repair analysis

This notebook analyzes the exact Stage 8 outputs. Hidden-center construction and event tracing live only in the behavior-preserving source implementation.

In [ ]:
from src.io import PipelinePaths, load_optional_csv, load_stage8_outputs
from src.diagnostics.invariants import validate_tracks

paths = PipelinePaths.discover()
outputs = load_stage8_outputs(paths=paths)
onsets = load_optional_csv(paths.stage8_stitching / 'merge_onsets.csv')
trajectories = load_optional_csv(paths.stage8_stitching / 'merge_center_trajectories.csv')
repairs = load_optional_csv(paths.stage8_stitching / 'merge_track_repairs.csv')
print('Track invariant warnings:', validate_tracks(outputs.tracks))
print('Onsets:', len(onsets), 'virtual centers:', len(trajectories), 'remaps:', len(repairs))

In [ ]:
onsets.head()

In [ ]:
import matplotlib.pyplot as plt

if trajectories.empty:
    print('No accepted merge trajectory is present in this Stage 8 output.')
else:
    event_column = 'event_id' if 'event_id' in trajectories else None
    selected = trajectories
    if event_column is not None:
        selected = trajectories[trajectories[event_column] == trajectories[event_column].iloc[0]]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
    for coordinate, axis in zip(['z', 'y', 'x'], axes):
        for track_id, rows in selected.groupby('parent_track_id'):
            axis.plot(rows.frame, rows[f'center_{coordinate}'], marker='o', label=f'track {track_id}')
        axis.set_title(f'virtual {coordinate} center')
        axis.set_xlabel('frame')
    axes[0].legend()
    plt.tight_layout()